In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/home/pwiesenbach/BertGCN")

from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer
from clinic_datasets import CleanClinicDataset

In [3]:
MODELTYPE = "deepset/gbert-base"
DATASET = "CARDIODE400_main"
DATASETPATH =  Path("/home/pwiesenbach/BertGCN/data") / f"ind.{DATASET}"
#DATASETFILE = Path("/home/pwiesenbach/BertGCN/data") / "CARDIODE400.json"

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

tokenizer = AutoTokenizer.from_pretrained(MODELTYPE)

train_dataset_file = Path("/home/pwiesenbach/BertGCN/data") / "csc_train_bert.json"
test_dataset_file = Path("/home/pwiesenbach/BertGCN/data") / "csc_test_bert.json"

with open(train_dataset_file, "rb") as f:
    train_dataset = pickle.load(f)

with open(test_dataset_file, "rb") as f:
    test_dataset = pickle.load(f)

In [4]:
mixfactor = 0.5
ig_gcn_bert_path = f"/home/pwiesenbach/BertGCN/models/gcn/{mixfactor}/ig_attrs_gcn_CSC.npz"
ig_gcn_bert_values = np.load(ig_gcn_bert_path)["arr_0"]

In [5]:
ig_gcn_bert_values.shape

(21891, 116871)

In [20]:
top_n_interpret = 3
top_ig_gcn_bert_values = np.argpartition(ig_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_bert_values.shape

(21891, 3)

In [21]:
random.seed(0)
idx = np.arange(len(train_dataset))
random.shuffle(idx)
train_idx, val_idx = idx[: int(len(idx) * 0.9)], idx[int(len(idx) * 0.9) :]

train_len = len(train_idx)
val_len = len(val_idx)
test_len = len(test_dataset)


def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return x
    
train_len, val_len, test_len

(85482, 9498, 21891)

In [22]:
top_ig_gcn_bert_values = np.vectorize(map_to_idx)(top_ig_gcn_bert_values)

In [9]:
def convert_to_64bit_indices(A):
    A.indptr = np.array(A.indptr, copy=False, dtype=np.int64)
    A.indices = np.array(A.indices, copy=False, dtype=np.int64)
    return A

adj = convert_to_64bit_indices(adj)

first_order_adj = adj @ adj
first_order_adj = first_order_adj.toarray()

In [23]:
top_n_input = 10
top_input_test_rel_nodes = np.argpartition(first_order_adj[test_mask][:, doc_mask], -top_n_input)[:, -top_n_input:]
top_input_test_rel_nodes = np.vectorize(map_to_idx)(top_input_test_rel_nodes)
top_input_test_rel_nodes.max(), top_input_test_rel_nodes.shape

(116870, (21891, 10))

In [24]:
s = 0
for a, b in zip(top_input_test_rel_nodes, top_ig_gcn_bert_values):
    s += len(np.intersect1d(a, b))
print(s)

19227


In [30]:
input_only, interpret_only, both = list(), list(), list()
for a, b in zip(top_input_test_rel_nodes, top_ig_gcn_bert_values):
    input_only.append(np.setdiff1d(a, b))
    interpret_only.append(np.setdiff1d(b, a))
    both.append(np.intersect1d(b, a))
diff_df = pd.DataFrame({"interpret only": interpret_only, "input_only": input_only, "both": both}, 
                       index=range(train_len + val_len, train_len + val_len + test_len)
                      )
diff_df

,interpret only,input_only,both
94980,[],"[22938, 36536, 41633, 42423, 94447, 95145, 95656]","[72241, 94980, 111203]"
94981,"[34355, 43301, 94981]","[38482, 38941, 49706, 56332, 87422, 92404, 929...",[]
94982,[114495],"[24911, 26335, 26391, 26404, 26420, 50154, 950...","[62966, 94982]"
94983,"[5048, 39961, 68636]","[24968, 31868, 45867, 50155, 50177, 62965, 646...",[]
94984,"[81961, 94456]","[34263, 50197, 54373, 57990, 62647, 87833, 929...",[94984]
...,...,...,...
116866,"[88315, 116866, 116870]","[13653, 30723, 54851, 54867, 54920, 54958, 790...",[]
116867,"[16865, 36407, 116867]","[9100, 20944, 51365, 65694, 73603, 85217, 9391...",[]
116868,"[52063, 116868]","[6038, 11512, 36852, 60613, 61703, 96868, 1003...",[85235]
116869,"[58751, 73945, 116869]","[16510, 22524, 22527, 39226, 47158, 62526, 632...",[]


In [31]:
diff_df[diff_df['interpret only'].map(len) == 3]

,interpret only,input_only,both
94981,"[34355, 43301, 94981]","[38482, 38941, 49706, 56332, 87422, 92404, 929...",[]
94983,"[5048, 39961, 68636]","[24968, 31868, 45867, 50155, 50177, 62965, 646...",[]
94986,"[15676, 72144, 94986]","[16, 6292, 12362, 15753, 24338, 49165, 58268, ...",[]
94991,"[48296, 61456, 94991]","[38482, 39326, 52989, 64889, 65760, 92404, 929...",[]
94993,"[6990, 9477, 83630]","[43343, 55187, 56091, 60921, 91074, 91077, 949...",[]
...,...,...,...
116865,"[75023, 91941, 116865]","[3264, 5931, 5932, 22524, 22527, 31700, 80058,...",[]
116866,"[88315, 116866, 116870]","[13653, 30723, 54851, 54867, 54920, 54958, 790...",[]
116867,"[16865, 36407, 116867]","[9100, 20944, 51365, 65694, 73603, 85217, 9391...",[]
116869,"[58751, 73945, 116869]","[16510, 22524, 22527, 39226, 47158, 62526, 632...",[]
